# 12 · Hawkeye's precision
*One arrow, one target: fewer files, smarter layout, cheaper queries.*

The briefings run on a few thousand rows. Real field data does not. Hawkeye generates **500,000 incidents**, stores them twice (once as they come, once laid out for the question he asks most) and compares what the same query costs.
Measure with the **Query Profile** (query history → a query → *See query profile*), not with your feelings: durations in a notebook are noisy, so this notebook compares results and plans, and shows times only for your eyes.

| Concept | Where you meet it |
|---|---|
| Data skipping and file layout | the next section |
| Liquid clustering (`CLUSTER BY`) | Step 3 |
| Measuring with results, times and the Query Profile | Step 4 |
| Broadcast joins | Step 5 |
| Predictive optimization, partitioning | "Other habits" at the end (not run) |

Deep dive: [docs/12-hawkeyes-precision.md](../../docs/12-hawkeyes-precision.md)

**Needs:** chapter 01 (schema `shield.ops`, the `landing` volumes); for Step 5 `shield.silver.locations` from chapter 05 or 06; serverless compute. The notebook writes about 120 MB of JSON into the staging volume and two tables into `shield.ops`; the clean-up removes both.
The **Expected** counts come from the generator with the seed in Step 1 (`seed=7`, no defects) and assume a UTC session time zone (the serverless default). **Durations, file counts and sizes cannot be predicted and nothing here asserts them.**

## Data skipping and liquid clustering

**The idea:** Delta keeps the smallest and largest value of a column for every data file. A query with `WHERE location_id = 'L003'` can skip every file whose range cannot contain `L003`. That is **data skipping**.
It only works when each file holds a narrow range, so similar rows must sit together. How rows are laid out in files decides how many files a query has to read. **Liquid clustering** is the Delta table setting that does the arranging: you name the columns you filter by (`CLUSTER BY (...)`), and `OPTIMIZE` groups the rows of those columns into files. You can change the columns later with `ALTER TABLE ... CLUSTER BY`.
It replaces hand-tuned partitioning and `ZORDER`, which cannot be combined with it.

```
 unorganised table: every file holds every location and every month
   file 1 [L001..L010, Jan..Dec]   file 2 [L001..L010, Jan..Dec]   file 3 [L001..L010, Jan..Dec]
   WHERE location_id = 'L003' AND March  ->  all three files might match: read all of them

 clustered by (location_id, reported_ts)
   file 1 [L001..L003, Jan..Jun]   file 2 [L003..L006, Jan..Dec]   file 3 [L007..L010, ...]
   same query  ->  files whose range cannot match are skipped
```

**How this notebook implements it**

* Step 1 generates 500,000 incidents in 50 JSON files. Step 2 writes them as `shield.ops.incidents_plain`, with no layout at all: each file holds a mixture of everything.
* Step 3 copies the table into `shield.ops.incidents_clustered` with `CLUSTER BY (location_id, reported_ts)` and runs `OPTIMIZE`.
* Step 4 runs one query on both tables: *incidents of location `L003` in March 2025*. The answers must be identical; what differs is how much data was read.

**Why it helps:** fewer files read means fewer bytes, less compute, lower cost.

**An honest warning:** 500,000 rows are about 120 MB of JSON and probably only tens of megabytes as Parquet, so both tables will end up with only a handful of files, maybe one. With a single file there is nothing to skip, and you will see no pruning at all. That is a real lesson (layout pays off on big tables), not a failure. Raise `INCIDENTS` in Step 1 to 5,000,000 to make the effect visible (Try it in the doc).
Both tables are Unity Catalog *managed* tables, for which Databricks already tunes file sizes automatically, and predictive optimization may be running `OPTIMIZE` for you in the background (see "Other habits").

**Not covered:** `CLUSTER BY AUTO` (Databricks picks the keys), choosing keys for a real workload, partitioning.

## Setup

In [ ]:
import os
import shutil
import sys
import time
from pathlib import Path

from pyspark.sql import functions as F

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
from shield_data.generate import generate

INCIDENTS = 500_000
BIG = Path("/Volumes/shield/landing/staging/big")


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected.
    An expected value of None means there is no single right answer: compare by eye (shown as –)."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    mark = lambda got, want: "–" if want is None else ("✅" if got == want else "❌")
    rows = [(what, fmt(got), "–" if want is None else fmt(want), mark(got, want)) for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


schemas = {row[0] for row in spark.sql("SHOW SCHEMAS IN shield").collect()}
assert "ops" in schemas, "Run notebook 01 first. Missing schema: shield.ops"
assert Path("/Volumes/shield/landing/staging").is_dir(), "Run notebook 01 first. Missing volume: shield.landing.staging"

## 1 · The data

### Step 1 · Generate 500,000 incidents
**Goal:** a table big enough for layout to matter, without defects or duplicates so the numbers are easy to check.

| Input | Value |
|---|---|
| `generate(BIG, incidents=500_000, pings=1000, seed=7, dirty=0.0, batches=50)` | 50 JSON files in `/Volumes/shield/landing/staging/big/incident_reports` |

**Expected:** 500,000 rows with 500,000 distinct `incident_id`s in 50 files; location `L003` is the largest with 50,539 rows. The generator runs once: if the folder exists it is reused (delete it to regenerate).
The last 20 % of the rows carry the extra field `source_channel` (schema drift from chapter 03), so the read below has that column too.

In [ ]:
if not (BIG / "incident_reports").exists():
    generate(BIG, incidents=INCIDENTS, pings=1000, seed=7, dirty=0.0, batches=50)

incidents_raw = spark.read.json(f"{BIG}/incident_reports").withColumn("reported_ts", F.to_timestamp("reported_at"))
summary = incidents_raw.agg(
    F.count("*").alias("rows"),
    F.countDistinct("incident_id").alias("distinct_ids"),
    F.sum(F.when(F.col("location_id") == "L003", 1).otherwise(0)).alias("l003_rows"),
).first()

show_checks([
    ("incident files", len(list((BIG / "incident_reports").glob("*"))), 50),
    ("incident rows", summary["rows"], 500_000),
    ("distinct incident_id", summary["distinct_ids"], 500_000),
    ("rows of location L003", summary["l003_rows"], 50_539),
])

## 2 · Two layouts

### Step 2 · Baseline: the unorganised table
**Goal:** write the data as it came, with no layout. This is the table everything else is compared with.

**Expected:** `shield.ops.incidents_plain` has 500,000 rows. `numFiles` and `sizeInBytes` have no single right answer (about one file per input file at most, a few tens of megabytes; not verified), so they are shown, not checked.
Re-running the cell overwrites the table with the same data.

In [ ]:
incidents_raw.write.mode("overwrite").saveAsTable("shield.ops.incidents_plain")

plain_detail = spark.sql("DESCRIBE DETAIL shield.ops.incidents_plain").select("numFiles", "sizeInBytes").first()
show_checks([
    ("rows in incidents_plain", spark.table("shield.ops.incidents_plain").count(), 500_000),
    ("files in incidents_plain", plain_detail["numFiles"], None),
    ("size in bytes", plain_detail["sizeInBytes"], None),
])

### Step 3 · Liquid clustering on the columns you filter by
**Goal:** the same rows, laid out by `location_id` and `reported_ts`.

**Expected:** `shield.ops.incidents_clustered` has the same 500,000 rows and reports `clusteringColumns = [location_id, reported_ts]`. The number of files is likely a handful, probably not more than the plain table's (not verified); it is shown, not checked.
`OPTIMIZE` is what clusters the data. Re-running the cell recreates the table.

In [ ]:
spark.sql(
    "CREATE OR REPLACE TABLE shield.ops.incidents_clustered CLUSTER BY (location_id, reported_ts) "
    "AS SELECT * FROM shield.ops.incidents_plain"
)
spark.sql("OPTIMIZE shield.ops.incidents_clustered")

clustered_detail = spark.sql("DESCRIBE DETAIL shield.ops.incidents_clustered").select("numFiles", "sizeInBytes", "clusteringColumns").first()
show_checks([
    ("rows in incidents_clustered", spark.table("shield.ops.incidents_clustered").count(), 500_000),
    ("clustering columns", list(clustered_detail["clusteringColumns"]), ["location_id", "reported_ts"]),
    ("files in incidents_clustered", clustered_detail["numFiles"], None),
    ("size in bytes", clustered_detail["sizeInBytes"], None),
])

## 3 · Measure

### Step 4 · Same query, two layouts
**Goal:** prove the layouts give the same answer, then compare what they cost.

| Input | Value |
|---|---|
| query | incidents of location `L003` reported in March 2025: count and sum of casualties |
| tables | `shield.ops.incidents_plain` and `shield.ops.incidents_clustered` |

**Expected:** both tables return `n = 4,184` and `casualties = 2,515`. The times have no right answer: each query runs twice and the second timing is shown, so warm-up does not favour the table that goes second.
Open both queries in **Query Profile** and compare **files read / files pruned** and **bytes read**. With a handful of files you may see no pruning (see the warning above); with more data the clustered table should read fewer files (not verified).

In [ ]:
QUERY = """
SELECT COUNT(*) AS n, SUM(casualties) AS casualties
FROM {t}
WHERE location_id = 'L003' AND reported_ts >= '2025-03-01' AND reported_ts < '2025-04-01'
"""

answers, seconds = {}, {}
for table in ["shield.ops.incidents_plain", "shield.ops.incidents_clustered"]:
    for _ in range(2):  # the second run is the one we time
        start = time.time()
        answers[table] = tuple(spark.sql(QUERY.format(t=table)).first())
        seconds[table] = round(time.time() - start, 2)

show_checks([
    ("plain: (n, casualties)", answers["shield.ops.incidents_plain"], (4_184, 2_515)),
    ("clustered: (n, casualties)", answers["shield.ops.incidents_clustered"], (4_184, 2_515)),
    ("plain: seconds", seconds["shield.ops.incidents_plain"], None),
    ("clustered: seconds", seconds["shield.ops.incidents_clustered"], None),
])

### Step 5 · Join a small table with a broadcast
**Goal:** add the city to every incident without shuffling the 500,000-row side. A **broadcast join** sends the small table (10 locations) to every worker, so the big table stays where it is.

**Expected:** the joined result has 500,000 rows (every incident has a known location) and its plan contains a broadcast join. On Photon the node may be named `PhotonBroadcastHashJoin`; the check looks for the word "Broadcast". Plan text and node names are not verified.
Needs `shield.silver.locations` (chapter 05 or 06).

In [ ]:
assert spark.catalog.tableExists("shield.silver.locations"), "Run notebook 05 or 06 first. Missing table: shield.silver.locations"

locations_small = spark.table("shield.silver.locations")
incidents_with_city = spark.table("shield.ops.incidents_plain").join(F.broadcast(locations_small), "location_id")
incidents_with_city.createOrReplaceTempView("incidents_with_city")

plan_text = "\n".join(row[0] for row in spark.sql("EXPLAIN SELECT * FROM incidents_with_city").collect())
print(plan_text)
show_checks([
    ("joined rows", incidents_with_city.count(), 500_000),
    ("plan contains a broadcast join", "Broadcast" in plan_text, True),
])

## Other habits worth having (not run)

* Select only the columns you need; filter early. Avoid `collect()` and `toPandas()` on big tables.
* Keep joins on the same type and key; broadcast small dimensions (`F.broadcast`); filter before joining.
* **Predictive optimization** runs `OPTIMIZE`, `VACUUM` and `ANALYZE` on Unity Catalog managed tables for you. Learn says it is on by default for accounts created on or after 11 November 2024 and is being enabled for existing accounts gradually, expected to finish by August 2026. The Learn page "Predictive optimization for Unity Catalog managed tables" explains how to verify whether it is enabled for your account; check, don't assume.
* **Partitioning** is for very large tables with a low-cardinality column that every query filters on (for example a date). Over-partitioning creates small files; Databricks recommends liquid clustering for new tables.
* Prefer serverless or right-sized job compute; stop interactive compute when idle.

## 👀 In the workspace

* **Query history** (sidebar) → one of the Step 4 queries → **See query profile**: the scan node shows files read, files pruned and bytes read. Compare the two tables.
* **Catalog** → `shield` → `ops` → `incidents_clustered` → **Details**: the clustering columns; the **History** tab shows the `OPTIMIZE` run.

## 🧹 Clean up

The next cells drop the two tables and delete the generated JSON files (about 120 MB). Re-running the notebook generates them again, which takes a minute or two. `shield.ops` is chapter 01's schema and stays. No later chapter uses these tables.

In [ ]:
spark.sql("DROP TABLE IF EXISTS shield.ops.incidents_plain")
spark.sql("DROP TABLE IF EXISTS shield.ops.incidents_clustered")
spark.catalog.dropTempView("incidents_with_city")
shutil.rmtree(BIG, ignore_errors=True)

show_checks([
    ("incidents_plain still exists", spark.catalog.tableExists("shield.ops.incidents_plain"), False),
    ("incidents_clustered still exists", spark.catalog.tableExists("shield.ops.incidents_clustered"), False),
    ("generated files still exist", BIG.exists(), False),
])

## 🎯 Recap

* Data skipping reads only the files whose min/max range can match; layout decides how many that is. Liquid clustering lays rows out by the columns you filter by.
* Both layouts returned the same answer (4,184 incidents, 2,515 casualties for `L003` in March). What differs is files read and bytes scanned, and you read that in the Query Profile, not from a stopwatch.
* At 500,000 rows the effect can be small or invisible; it pays off as tables grow. A broadcast join keeps the big side in place.

Mini-projects: [docs/12-hawkeyes-precision.md](../../docs/12-hawkeyes-precision.md#try-it-yourself)

> 💡 Next: chapter 13 turns the platform into code with Bicep and bundles.